In [1]:
import os
import json
import math
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from numpy.linalg import norm
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy.stats import t
import shap

from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb

# opções: 'LOGREG', 'XGB_RANKER', 'LGBM_RANKER'
MODELO_ATUAL = 'LGBM_RANKER' 

NOME_BASE = 'wikidata'
SPLITS = [1, 2, 3, 4, 5]

"""
RELATION_STATS = {
    'conjugue': {'mean': -4.42, 'std': 10.45},
    'filho': {'mean': -35.26, 'std': 9.3},
    'irmao': {'mean': -0.48, 'std': 9.64},
    'mae': {'mean': 28.55, 'std': 7.6},
    'pai': {'mean': 34.54, 'std': 9.1},
    'parente': {'mean': -3.05, 'std': 49.08}
}
"""

PARENSTESCO_COLS = [
    "is_conjugue", "is_filho", "is_irmao", "is_mae", "is_pai", "is_parente"
]

EMB_DIRS = {
    'profissao': ('occupation_embeddings_br', 'Occupation: ', 'emb_Profissoes_base'),
    'endereço': ('residence_embeddings_br', 'Residence: ', 'emb_Enderecos_base'),
    'local_nascimento': ('birthplace_embeddings_br', 'Birthplace: ', 'emb_Locais_Nascimento_base'),
    'nacionalidade': ('nationality_embeddings_br', 'Nationality: ', 'emb_Nacionalidades_base')
}

os.makedirs("results", exist_ok=True)

In [2]:
embeddings_cache = {}

for campo, (folder, prefixo, col_name) in EMB_DIRS.items():
    embeddings_cache[campo] = {}
    if os.path.exists(folder):
        dataset = pq.ParquetDataset(folder)
        df_emb = dataset.read().to_pandas()
        col_text = [c for c in df_emb.columns if not c.startswith('emb_')][0]
        
        for _, row in df_emb.iterrows():
            embeddings_cache[campo][row[col_text]] = np.array(row[col_name], dtype=np.float32)

print("Embeddings carregados para a RAM.")

Embeddings carregados para a RAM.


In [3]:
def _get_year(data_str):
    try:
        if pd.isna(data_str) or str(data_str).lower() == 'nan': return None
        return int(str(data_str).strip()[:4])
    except: return None

def calc_max_cosseno_vetorizado(lista1, lista2, campo):
    if not lista1 or not lista2: return np.nan
    prefix = EMB_DIRS[campo][1]
    
    embs1 = [embeddings_cache[campo].get(f"{prefix}{i}") for i in lista1]
    embs2 = [embeddings_cache[campo].get(f"{prefix}{i}") for i in lista2]
    
    embs1 = [e for e in embs1 if e is not None]
    embs2 = [e for e in embs2 if e is not None]
    
    if not embs1 or not embs2: return np.nan
    
    # 1. Converte para array e substitui qualquer NaN ou Inf que já venha do parquet por 0.0
    A = np.nan_to_num(np.array(embs1), nan=0.0, posinf=0.0, neginf=0.0)
    B = np.nan_to_num(np.array(embs2), nan=0.0, posinf=0.0, neginf=0.0)
    
    norm_A = np.linalg.norm(A, axis=1, keepdims=True)
    norm_B = np.linalg.norm(B, axis=1, keepdims=True)
    
    # 2. Usa np.clip para garantir que NENHUM valor seja menor que 1e-10
    # Isso é mais seguro que 'norm == 0', pois protege contra floats minúsculos (ex: 1e-300)
    norm_A = np.clip(norm_A, a_min=1e-10, a_max=None)
    norm_B = np.clip(norm_B, a_min=1e-10, a_max=None)
    
    # 3. Calcula o cosseno
    cossenos = np.dot(A / norm_A, (B / norm_B).T)
    
    # 4. Última barreira: limpa o resultado do produto escalar limitando entre -1 e 1 (regras do cosseno)
    cossenos = np.nan_to_num(cossenos, nan=0.0)
    cossenos = np.clip(cossenos, a_min=-1.0, a_max=1.0)
    
    return float(np.max(cossenos))

def match_exato_lista(lista1, lista2):
    if not lista1 or not lista2: return 0
    set1 = set(str(x).lower().strip() for x in lista1)
    set2 = set(str(x).lower().strip() for x in lista2)
    return 1 if len(set1.intersection(set2)) > 0 else 0

def extrair_features(p1, p2, parentesco, modelo_atual):
    features = {}
    
    for col in PARENSTESCO_COLS: features[col] = 0
    parentesco_limpo = f"is_{parentesco.lower()}"
    if parentesco_limpo in features:
        features[parentesco_limpo] = 1

    y_ctx = _get_year(p1.get('data_nascimento'))
    y_cand = _get_year(p2.get('data_nascimento'))
    diff_idade = (y_cand - y_ctx) if (y_ctx and y_cand) else None
    
    stats = RELATION_STATS.get(parentesco.lower())

    if diff_idade is not None:
        graus_liberdade = stats['df']
        localizacao = stats['loc']
        escala = stats['scale']
        
        altura = t.pdf(diff_idade, df=graus_liberdade, loc=localizacao, scale=escala)
        altura_max = t.pdf(localizacao, df=graus_liberdade, loc=localizacao, scale=escala)
        
        features['age_score'] = float(altura / altura_max) if altura_max > 0 else 0.0
    else:
        features['age_score'] = np.nan

    # distância de embeddings vetorizada
    features['prof_score_ml_base'] = calc_max_cosseno_vetorizado(p1.get('profissao', []), p2.get('profissao', []), 'profissao')
    features['address_score_ml_base'] = calc_max_cosseno_vetorizado(p1.get('endereço', []), p2.get('endereço', []), 'endereço')
    features['nasc_score_ml_base'] = calc_max_cosseno_vetorizado(p1.get('local_nascimento', []), p2.get('local_nascimento', []), 'local_nascimento')
    features['nac_score_ml_base'] = calc_max_cosseno_vetorizado(p1.get('nacionalidade', []), p2.get('nacionalidade', []), 'nacionalidade')

    # match exato
    features['match_exato_nac'] = match_exato_lista(p1.get('nacionalidade', []), p2.get('nacionalidade', []))
    features['match_exato_nasc'] = match_exato_lista(p1.get('local_nascimento', []), p2.get('local_nascimento', []))
    features['match_exato_res'] = match_exato_lista(p1.get('endereço', []), p2.get('endereço', []))

    # missing com nulos para regressão logística
    if modelo_atual == 'LOGREG':
        features['missing_idade'] = 1 if np.isnan(features['age_score']) else 0
        features['missing_profissao'] = 1 if np.isnan(features['prof_score_ml_base']) else 0
        features['missing_endereco'] = 1 if np.isnan(features['address_score_ml_base']) else 0
        features['missing_nascimento'] = 1 if np.isnan(features['nasc_score_ml_base']) else 0
        features['missing_nacionalidade'] = 1 if np.isnan(features['nac_score_ml_base']) else 0
        
        for col in ['age_score', 'prof_score_ml_base', 'address_score_ml_base', 'nasc_score_ml_base', 'nac_score_ml_base']:
            if np.isnan(features[col]): features[col] = 0.0

    return features

print("Funções preparadas.")

Funções preparadas.


In [4]:
def carregar_dataset_pairwise(filepath, modelo_atual):
    query_buffer = {} 
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            row = json.loads(line)
            
            pessoa_alvo = row.get('pessoa_ground_truth', row.get('Dados_pessoa_1')) 
            pessoa_contexto = row.get('pessoa_contexto', row.get('Dados_pessoa_2')) 
            candidato = row.get('candidato') 
            parentesco = row.get('parentesco', row.get('Parentesco'))
            label = row.get('label', row.get('Label'))
            
            feat = extrair_features(pessoa_contexto, candidato, parentesco, modelo_atual)
            
            qid = pessoa_alvo['id']
            if qid not in query_buffer:
                query_buffer[qid] = []
                
            query_buffer[qid].append({'feat': feat, 'label': label})
            
    LGBM_LIMIT = 9000 
    
    data_rows = []
    labels = []
    query_ids = []
    
    for qid, items in query_buffer.items():
        """
        if modelo_atual == 'LGBM_RANKER' and len(items) > LGBM_LIMIT:
            positivos = [item for item in items if item['label'] == 1]
            negativos = [item for item in items if item['label'] == 0]
            
            chunk_size = LGBM_LIMIT - len(positivos)
            if chunk_size < 1: chunk_size = 1 # fallback de segurança
            
            for i in range(0, len(negativos), chunk_size):
                sub_query_id = f"{qid}_part{i // chunk_size}"
                chunk_negativos = negativos[i : i + chunk_size]
                
                sub_query_items = positivos + chunk_negativos
                
                for item in sub_query_items:
                    data_rows.append(item['feat'])
                    labels.append(item['label'])
                    query_ids.append(sub_query_id)
        else:
            for item in items:
                data_rows.append(item['feat'])
                labels.append(item['label'])
                query_ids.append(qid)
        """
        for item in items:
                data_rows.append(item['feat'])
                labels.append(item['label'])
                query_ids.append(qid)
            
    df = pd.DataFrame(data_rows)
    df['label'] = labels
    df['query_id'] = query_ids
    
    # ordenar por query_id alfabeticamente para rankers
    df = df.sort_values('query_id').reset_index(drop=True)
    
    X = df.drop(['label', 'query_id'], axis=1)
    y = df['label'].values
    grupos = df.groupby('query_id').size().values
    
    return X, y, grupos

In [5]:
print(f"Iniciando treinamento com o modelo: {MODELO_ATUAL}")

PATH_MODELOS = "modelos_treinados"
if not os.path.exists(PATH_MODELOS):
    os.makedirs(PATH_MODELOS)

modelos_treinados = {}

for split in SPLITS:
    print(f"\nTreinando Split {split}...")

    with open(f"splits/stats_split{split}.json", 'r') as f:
        RELATION_STATS = json.load(f)

    train_file = f"splits/{NOME_BASE}_split{split}_train.jsonl"
    valid_file = f"splits/{NOME_BASE}_split{split}_validation.jsonl"
    
    X_train, y_train, group_train = carregar_dataset_pairwise(train_file, MODELO_ATUAL)
    X_valid, y_valid, group_valid = carregar_dataset_pairwise(valid_file, MODELO_ATUAL)
    
    if MODELO_ATUAL == 'XGB_RANKER':
        model = xgb.XGBRanker(
            objective="rank:ndcg",
            eval_metric="ndcg@10",
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            random_state=42,
            n_jobs=-1
        )
        
        model.fit(
            X_train, y_train, group=group_train,
            eval_set=[(X_valid, y_valid)], eval_group=[group_valid],
            verbose=20
        )

        print(f"Split {split} pronto.")
        
    elif MODELO_ATUAL == 'LGBM_RANKER':
        model = lgb.LGBMRanker(
            objective="lambdarank",
            metric="ndcg",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1
        )

        model.fit(
            X_train,
            y_train,
            group=group_train,
            eval_set=[(X_valid, y_valid)],
            eval_group=[group_valid],
            eval_at=[1, 5, 10],
            callbacks=[
                lgb.log_evaluation(period=20)
            ]
        )

        print(f"Split {split} pronto.")

    elif MODELO_ATUAL == 'LOGREG':
        n_pos = (y_train == 1).sum()
        n_neg = (y_train == 0).sum()
        cw = {0: 1.0, 1: (n_neg / n_pos)} if n_pos > 0 else 'balanced'
        
        logreg_model = LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            class_weight=cw,
            n_jobs=-1,
            random_state=42
        )
        
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", logreg_model)
        ])
        
        model.fit(X_train, y_train)
    
    nome_arquivo = f"{PATH_MODELOS}/{MODELO_ATUAL}_split{split}.joblib"
    joblib.dump(model, nome_arquivo)
    print(f"Modelo salvo em: {nome_arquivo}")

    modelos_treinados[split] = model

Iniciando treinamento com o modelo: LGBM_RANKER

Treinando Split 1...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021213 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1278
[LightGBM] [Info] Number of data points in the train set: 900515, number of used features: 14
[20]	valid_0's ndcg@1: 0.881027	valid_0's ndcg@5: 0.938715	valid_0's ndcg@10: 0.942423
[40]	valid_0's ndcg@1: 0.883797	valid_0's ndcg@5: 0.939962	valid_0's ndcg@10: 0.943581
[60]	valid_0's ndcg@1: 0.885317	valid_0's ndcg@5: 0.940725	valid_0's ndcg@10: 0.944359
[80]	valid_0's ndcg@1: 0.885426	valid_0's ndcg@5: 0.941043	valid_0's ndcg@10: 0.944633
[100]	valid_0's ndcg@1: 0.886132	valid_0's ndcg@5: 0.941488	valid_0's ndcg@10: 0.945087
[120]	valid_0's ndcg@1: 0.886838	valid_0's ndcg@5: 0.941799	valid_0's ndcg@10: 0.945396
[140]	valid_0's ndcg@1: 0.887924	valid_0's ndcg@5: 0

In [ ]:
resultados_por_split = []
resultados_tabela_shap = []

def calcular_mrr(posicao):
    return 1.0 / posicao if posicao > 0 else 0.0

def lgbm_explainer(df_test, split, features, model_lgbm):
    explainer = shap.TreeExplainer(model_lgbm)
    
    X = df_test[features]

    shap_values = explainer.shap_values(X)
    
    # 1. Se retornar uma lista (versões antigas do SHAP com multiclasse/ranker)
    if isinstance(shap_values, list):
        # Pega a classe positiva (índice 1) se existir, senão pega o único elemento
        shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    
    # Converte para array numpy por segurança
    shap_values = np.array(shap_values)
    
    # 2. Se retornar um array 3D (amostras, features, classes) - versões mais novas
    if len(shap_values.shape) == 3:
        # Extrai a última dimensão (que geralmente representa a classe positiva/score principal)
        shap_values = shap_values[:, :, -1]
        
    importance = np.nanmean(np.abs(shap_values), axis=0)
    
    df_importance = pd.DataFrame({
        "feature": features,
        "importance": importance
    }).sort_values("importance", ascending=False)
    
    total = df_importance["importance"].sum()
    if total == 0: total = 1e-10 # Evita divisão por zero
    
    feature_groups = {
        "address": [
            'address_score_ml_base', 'nasc_score_ml_base', 'nac_score_ml_base', 
            'match_exato_nac', 'match_exato_nasc', 'match_exato_res'
        ],
        "age": ['age_score'],
        "profession": ['prof_score_ml_base'],
        "relationship_indicators": [
            'is_conjugue', 'is_filho', 'is_irmao', 'is_mae', 'is_pai', 'is_parente'
        ],
        "name_mother_father": [] # Vazio pois não há feature extraída disso
    }    
    
    group_importance = []
    for group, feats in feature_groups.items():
        imp = df_importance[df_importance["feature"].isin(feats)]["importance"].sum()
        group_importance.append({
            "group": group,
            "importance": imp
        })
    
    df_group = pd.DataFrame(group_importance)
    df_group["relative_importance"] = (100 * (df_group["importance"] / total))
    
    os.makedirs(f"results/{split}", exist_ok=True)
    output_file = f"results/{split}/lgbm_explainer.csv"
    df_group.to_csv(output_file, index=False)
    
    linha_resultado = {"Split": split}
    for _, row in df_group.iterrows():
        linha_resultado[row['group']] = row['relative_importance']
        
    return linha_resultado

for split in SPLITS:
    print(f"\nProcessando Inferência - Split {split}...")

    with open(f"splits/stats_split{split}.json", 'r') as f:
        RELATION_STATS = json.load(f)

    test_file = f"splits/{NOME_BASE}_split{split}_test.jsonl"
    split_results = []
    
    todas_features_lote = []
    queries_metadata = []
    
    with open(test_file, 'r', encoding='utf-8') as f:
        for line in tqdm(f, desc=f"Extraindo Features (Split {split})"):
            reg = json.loads(line)
            
            pessoa_alvo = reg.get('pessoa_ground_truth', reg.get('Dados_pessoa_1')) 
            pessoa_contexto = reg.get('candidato_correto', reg.get('Dados_pessoa_2')) 
            homonimos = reg.get('lista_homonimos', reg.get('Lista_homonimos'))
            parentesco = reg.get('parentesco', reg.get('Parentesco'))
            
            meta = {
                "id_pessoa_1": pessoa_alvo.get('id'),
                "Parentesco": parentesco,
                "qtd_homonimos": len(homonimos),
                "candidatos": []
            }
            
            for h in homonimos:
                is_correto = (str(h.get('id')) == str(pessoa_alvo.get('id')))
                feat = extrair_features(pessoa_contexto, h, parentesco, MODELO_ATUAL)
                
                todas_features_lote.append(feat)
                meta["candidatos"].append({
                    "id": h.get('id'),
                    "is_correto": is_correto
                })
                
            queries_metadata.append(meta)
            
    # predição vetorizada
    print(f"Executando predição em lote para {len(todas_features_lote)} pares candidato-contexto...")
    df_teste_lote = pd.DataFrame(todas_features_lote)
    df_teste_lote = df_teste_lote[X_train.columns] # Alinhando com as features de treino
    
    if MODELO_ATUAL in ['XGB_RANKER', 'LGBM_RANKER']:
        scores_lote = modelos_treinados[split].predict(df_teste_lote)
        
        if MODELO_ATUAL == 'LGBM_RANKER':
            print("Calculando SHAP Values para o LGBM...")
            linha_shap = lgbm_explainer(df_teste_lote, split, X_train.columns.tolist(), modelos_treinados[split])
            resultados_tabela_shap.append(linha_shap)
            
    elif MODELO_ATUAL == 'LOGREG':
        scores_lote = modelos_treinados[split].predict_proba(df_teste_lote)[:, 1]
        
    print("Calculando Recall e MRR...")
    cursor_score = 0 # aponta para o score atual na lista
    
    for meta in queries_metadata:
        resultados_ranking = []
        
        for cand in meta["candidatos"]:
            resultados_ranking.append((cand["id"], scores_lote[cursor_score], cand["is_correto"]))
            cursor_score += 1
            
        resultados_ranking.sort(key=lambda x: x[1], reverse=True)
        
        posicao_correta = -1
        for idx, (_, _, is_p2) in enumerate(resultados_ranking):
            if is_p2:
                posicao_correta = idx + 1
                break
                
        linha_resultado = {
            "id_pessoa_1": meta["id_pessoa_1"],
            "Parentesco": meta["Parentesco"],
            "MRR": calcular_mrr(posicao_correta)
        }
        
        for k in [1, 5, 10, 15, 20]:
            linha_resultado[f"Recall@{k}"] = 1.0 if (posicao_correta > 0 and posicao_correta <= k) else 0.0
            
        split_results.append(linha_resultado)
        
    df_metrics = pd.DataFrame(split_results)
    colunas_metricas = [col for col in df_metrics.columns if 'Recall' in col or 'MRR' in col]
    medias_split = df_metrics[colunas_metricas].mean().to_dict()
    medias_split['Split'] = split
    resultados_por_split.append(medias_split)

# Exibição dos Resultados de Performance
df_final = pd.DataFrame(resultados_por_split)

cols = ['Split', 'MRR', 'Recall@1', 'Recall@5', 'Recall@10', 'Recall@15', 'Recall@20']
df_final = df_final[cols]

media_geral = df_final.mean().to_dict()
media_geral['Split'] = 'Média'
df_final.loc[len(df_final)] = media_geral

print(f"\n{MODELO_ATUAL}: Resultados Processamento das bases para o artigo SBBD 2026")
display(df_final.round(4))

if MODELO_ATUAL == 'LGBM_RANKER' and len(resultados_tabela_shap) > 0:
    df_shap_final = pd.DataFrame(resultados_tabela_shap)
    
    cols_order = ['Split', 'address', 'age', 'profession', 'relationship_indicators', 'name_mother_father']
    df_shap_final = df_shap_final[cols_order]
    
    print("\n" + "="*85)
    print("LGBM Ranker Explainer".center(85))
    print("="*85)
    
    print("Split\taddress\t\tage\tprofession\trelationship_indicators\tname_mother_father")
    
    for _, row in df_shap_final.iterrows():
        print(f"{int(row['Split'])}\t{row['address']:.2f}%\t\t{row['age']:.2f}%\t{row['profession']:.2f}%\t\t{row['relationship_indicators']:.2f}%\t\t\t{row['name_mother_father']:.2f}%")


Processando Inferência - Split 1...


Extraindo Features (Split 1): 0it [00:00, ?it/s]

Executando predição em lote para 631791 pares candidato-contexto...
Calculando SHAP Values para o LGBM...
Calculando Recall e MRR...

Processando Inferência - Split 2...


Extraindo Features (Split 2): 0it [00:00, ?it/s]

Executando predição em lote para 596794 pares candidato-contexto...
Calculando SHAP Values para o LGBM...
Calculando Recall e MRR...

Processando Inferência - Split 3...


...

Executando predição em lote para 619856 pares candidato-contexto...
Calculando SHAP Values para o LGBM...
Calculando Recall e MRR...

Processando Inferência - Split 5...


Extraindo Features (Split 5): 0it [00:00, ?it/s]

Executando predição em lote para 632152 pares candidato-contexto...
Calculando SHAP Values para o LGBM...
Calculando Recall e MRR...

LGBM_RANKER: Resultados Processamento das bases para o artigo SBBD 2026


,Split,MRR,Recall@1,Recall@5,Recall@10,Recall@15,Recall@20
0,1,0.9290,0.8827,0.9847,0.9922,0.9946,0.9961
1,2,0.9276,0.8798,0.9851,0.9925,0.9949,0.9961
2,3,0.9283,0.8811,0.9847,0.9923,0.9947,0.9959
3,4,0.9285,0.8813,0.9853,0.9924,0.9946,0.9957
4,5,0.9276,0.8804,0.9844,0.9923,0.9946,0.9956
5,Média,0.9282,0.8811,0.9848,0.9923,0.9947,0.9959



                                LGBM Ranker Explainer                                
Split	address		age	profession	relationship_indicators	name_mother_father
1	15.39%		66.04%	13.84%		4.73%			0.00%
2	15.57%		64.86%	14.63%		4.94%			0.00%
3	14.71%		65.74%	14.54%		5.01%			0.00%
4	14.93%		66.25%	14.25%		4.56%			0.00%
5	13.44%		67.32%	14.86%		4.37%			0.00%
